# Estudio de Operador Circular y Recombinación Geométrica (Anexo D)

Este cuaderno contiene la evaluación experimental y el análisis del diseño factorial $2 \times 2$ correspondiente al **Anexo D** (*Operator Ablation and Architectural Progression*) de la memoria del TFG:

1. **Recombinación de escala ($\sigma$):** Aritmética ($(\sigma_1 + \sigma_2)/2$) vs. Geométrica ($\sqrt{\sigma_1 \sigma_2}$).
2. **Mutación circular:** Forced-Step ($\pm 1$, sin reposo) vs. Modular Skellam (con probabilidad de paso cero $\Delta x = 0$).

Los datos numéricos brutos se almacenan en formato dual:
- `data/factorial_study_data.npz`: Arrays compactos de NumPy con las pérdidas individuales por semilla.
- `data/factorial_study_results.json`: Resumen estadístico estructurado (mínimos, medias, desviaciones).


In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'results')


---
## 1. Carga de Datos desde Archivo NumPy (.npz)

Verificamos la carga directa de los arrays numéricos de pérdidas por semilla almacenados en `factorial_study_data.npz`.


In [ ]:
# Carga directa de arrays NumPy
npz_file = os.path.join(RESULTS_DIR, 'factorial_study_data.npz')
npz_data = np.load(npz_file)
print('Claves disponibles en el archivo .npz:')
for k in npz_data.files:
    arr = npz_data[k]
    print(f'  - {k:<35}: shape={arr.shape}, dtype={arr.dtype}')


---
## 2. Tabla de Resultados del Estudio Factorial (Tabla D.1 de la Memoria)

Evaluación sobre los 3 bancos de prueba heterogéneos de 22 dimensiones (Sphere, Coupled, Rastrigin) a lo largo de 10 semillas canónicas (20.000 evaluaciones por réplica).


In [ ]:
# Cargar y presentar resumen de métricas
with open(os.path.join(DATA_DIR, 'factorial_study_results.json'), 'r', encoding='utf-8') as f:
    results = json.load(f)

print(f"{'Variante':<25} | {'Benchmark':<12} | {'Min':<14} | {'Mean':<14} | {'Std':<14}")
print('-' * 85)
for var_name, b_dict in results.items():
    for b_key in ['sphere', 'coupled', 'rastrigin']:
        if b_key in b_dict:
            m = b_dict[b_key]
            print(f"{var_name:<25} | {b_key:<12} | {m['min']:<14.6e} | {m['mean']:<14.6e} | {m['std']:<14.6e}")


### Dinámica de Convergencia en Escala Logarítmica
El gráfico muestra la progresión de las variantes de diseño: la combinación de mutación Skellam y recombinación geométrica reduce el error en 7 órdenes de magnitud.


In [ ]:
display(Image(filename=os.path.join(RESULTS_DIR, 'step3_convergence_log.png')))


---
## 3. Conclusiones y Mecanismos Identificados (Anexo D)

1. **Sesgo de Inflación por Recombinación Aritmética:** Por la desigualdad de Jensen ($\frac{\sigma_1 + \sigma_2}{2} \ge \sqrt{\sigma_1 \sigma_2}$), promediar aritméticamente parámetros de escala log-normales infla sistemáticamente los pasos, impidiendo el enfriamiento adaptativo (bloqueo en error $2.805$).
2. **Suelo de Ruido en Mutación Circular Forzada:** Cuando se fuerza $\Delta x \in \{-1, +1\}$ sin posibilidad de reposo ($\Delta x = 0$), el algoritmo no puede fijar la solución óptima, estancándose en un error residual de $\approx 1.000$ (coste geodésico cuadrático de 1 paso).
3. **Sinergia Resolutiva:** La unión de la mutación modular de Skellam con la recombinación intermedia geométrica elimina ambos problemas simultáneamente, alcanzando una precisión de $6.225 \times 10^{-7}$.
